In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# cls_parser.pkl
COPY passwords.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
snowflake-connector-python

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from passwords import *
import snowflake.connector
from datetime import datetime

# constants
str_project = '20231010-gen-xii'
str_task = '13_payload_parsing'
str_subtask = 'payloads'

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# connect to snowflake
conn = snowflake.connector.connect(
    user=USERNAME,
    password=PASSWORD,
    account=ACCOUNT,
    warehouse=WAREHOUSE,
    database=DATABASE,
    schema=SCHEMA,
)

# query
str_query = """
select *
from 
raw.source_s3_scorehistory.PAYLOADS_PARSED_YESTERDAY
"""

# pull payloads
df = pd.read_sql(
    sql=str_query,
    con=conn,
)

# subset to gen 12
df = df[df['RESPONSE_MODEL_NAME'] == 'PRESTIGE-GEN-XII'].copy()

# write to s3
str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-pull-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  26.62kB
Step 1/8 : FROM python:3.9
 ---> 4b15bb967077
Step 2/8 : RUN apt-get update
 ---> Using cache
 ---> c59ecdb5b59b
Step 3/8 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 692397bbf273
Step 4/8 : COPY requirements.txt .
 ---> Using cache
 ---> 7bf24a45fb40
Step 5/8 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 8a07b30f4f17
Step 6/8 : COPY script.py .
 ---> 295e814db601
Step 7/8 : COPY passwords.py .
 ---> def497ebd8a5
Step 8/8 : CMD ["python3", "script.py"]
 ---> Running in 46809cdf3b63
Removing intermediate container 46809cdf3b63
 ---> d32a627c480d
Successfully built d32a627c480d
Successfully tagged genxii-pull-payloads:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pull-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pull-payloads]
4aa6123bc292: Preparing
92f722de9389: Preparing
c0c5e8244b9a: Preparing
faadf512c264: Preparing
bf86e0cc03ba: Preparing
69b638ea12af: Preparing
afe28ac5c5d1: Preparing
f3b460831925: Preparing
20e2f78dadaf: Preparing
e077e19b6682: Preparing
21e1c4948146: Preparing
68866beb2ed2: Preparing
e6e2ab10dba6: Preparing
0238a1790324: Preparing
afe28ac5c5d1: Waiting
f3b460831925: Waiting
20e2f78dadaf: Waiting
e077e19b6682: Waiting
21e1c4948146: Waiting
68866beb2ed2: Waiting
e6e2ab10dba6: Waiting
69b638ea12af: Waiting
c0c5e8244b9a: Layer already exists
4aa6123bc292: Layer already exists
bf86e0cc03ba: Layer already exists
faadf512c264: Layer already exists
69b638ea12af: Layer already exists
f3b460831925: Layer already exists
afe28ac5c5d1: Layer already exists
20e2f78dadaf: Layer already exists
e077e19b6682: Layer already exists
68866beb2ed2: Layer already exists
e6e2ab10dba6: Layer already exists
21e1c

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass